In [ ]:
# 01_eda.ipynb -- Era 3 (Nov 2024-present) SCADA EDA: violation + ramp-shock rate by
# hour, season, gen-mix, and corridor stress
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip
# !pip install lightgbm -q

import sys
sys.path.insert(0, "..")  # ML/Study2/ -- for features.py when run from notebooks/
import pandas as pd
import matplotlib.pyplot as plt

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)

df = f.drop_bad_days(scada)  # drops 2024-11-20, 2025-04-01 (1 slot), 2025-10-02 (63 slots)
df = f.add_datetime(df)
df = f.add_violation_label(df)
df = f.add_ramp_label(df)
df = f.add_time_features(df)

print("rows after dropping bad days:", len(df), "of", len(scada))
print("overall violation rate:", round(df["violation"].mean(), 4))
print("overall ramp rate:", round(df["ramp"].mean(), 4))

# --- Rate by hour of day ---
by_hour = df.groupby("hour")[["violation", "ramp"]].mean()
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
by_hour["violation"].plot(ax=axes[0], title="Frequency-violation rate by hour")
by_hour["ramp"].plot(ax=axes[1], title="Ramp-shock rate by hour", color="orange")
plt.tight_layout()
plt.savefig("era3_rate_by_hour.png")
print(by_hour.round(4))

# --- Rate by month (seasonality) ---
by_month = df.groupby("month")[["violation", "ramp"]].mean()
print("\n", by_month.round(4))

# --- Solar-hour vs non-solar-hour ---
print("\nviolation rate, solar vs non-solar hour:")
print(df.groupby("is_solar_hr")["violation"].mean().round(4))
print("ramp rate, solar vs non-solar hour:")
print(df.groupby("is_solar_hr")["ramp"].mean().round(4))

# --- Weekday vs weekend ---
print("\nviolation rate, weekday(0) vs weekend(1):")
print(df.groupby("is_weekend")["violation"].mean().round(4))

# --- Generation mix: RES share vs event rate ---
df["res_bin"] = pd.qcut(df["share_res_pct"], 5, duplicates="drop")
print("\nviolation/ramp rate by RES-share quintile (low -> high):")
print(df.groupby("res_bin", observed=True)[["violation", "ramp"]].mean().round(4))

# --- Corridor congestion: sum|ir_net| vs event rate ---
ir_abs_sum = df[f.CORRIDOR_COLS[:7]].abs().sum(axis=1)  # the 7 ir_* corridor cols
df["ir_bin"] = pd.qcut(ir_abs_sum, 5, duplicates="drop")
print("\nviolation/ramp rate by corridor-flow quintile (low -> high):")
print(df.groupby("ir_bin", observed=True)[["violation", "ramp"]].mean().round(4))

# --- Findings (verified 2026-07-11 against live study2_scada.csv, 56,892 usable slots
#     after dropping the 3 corrupted-file days) ---
#
# Overall base rates: violation 0.89%, ramp-shock 6.1% -- both rare-event but workable
# class balances (matches the 0.88% violation rate measured earlier in the roadmap).
#
# STRONG time-of-day pattern, and it lines up with solar ramp physics, not noise:
#   - Violations cluster 07:00-14:00, peaking at 13:00 (4.0%) and 08:00-09:00 (~3%) --
#     the mid-morning-to-early-afternoon window where solar output is both large and
#     fast-changing (cloud transients, ramp-up/plateau).
#   - Ramp-shocks cluster in two bands: 05:00-09:00 (sunrise ramp-up, peaking 36% at
#     06:00) and 17:00-20:00 (sunset ramp-down, up to 8.6%) -- the two times of day solar
#     generation changes fastest. This is a clean, physically-explainable signal, not an
#     artifact -- and it's the single strongest predictor a lead-time model should exploit
#     (confirmed in 03/04's feature importance: "hour" is the top feature for ramp-shock).
#   - Solar-hour violation rate (1.53%) is ~6x the non-solar-hour rate (0.26%).
#
# RES share vs violation rate is MONOTONIC and increasing: 0.33% in the lowest RES-share
# quintile up to 1.92% in the highest -- direct SCADA-resolution evidence for the
# project's central "rising RES share stresses the grid" thesis (Era 1 found a similar,
# weaker signal at daily/monthly resolution; this is the live, granular version of it).
#
# RES share vs ramp-shock rate goes the OTHER way in this simple quintile binning (8.3%
# in lowest quintile down to 2.8% in highest) -- flagged as a genuine, unresolved,
# counter-intuitive finding, not smoothed over: RES share is itself strongly seasonal
# (higher in summer), and month is independently a strong driver of ramp rate (Jan/Dec
# ~13-15%, Jul ~0.7%), so this crude binning is very likely confounded by season rather
# than showing a true RES effect. A cleaner month-controlled analysis is future work, not
# resolved here.
#
# Corridor-flow (ir_abs_sum) quintiles show no clean monotonic relationship with either
# event rate in this simple binning -- consistent with Era 2's daily-resolution finding
# that corridor flow's relationship to stress is real but not simply "more flow = more
# events" (it's corridors correcting stress, not causing it). The lead-time classifiers
# in 03/04 use the raw per-corridor columns rather than this aggregate, which is
# expected to capture more of that relationship than a single summed quintile can.
